# Examen ICCD753 Recuperación de Información

## 1. Objetivo

Desarrollar un sistema de recuperación de información sobre un corpus asignado. El sistema deberá indexar los documentos, recuperar y ordenar resultados mediante TF-IDF con similitud coseno y BM25, y exponer la búsqueda como un servicio web publicado en internet.

Se evaluará el servicio enviando consultas desde un cliente (p.ej. Postman) y contrastando los resultados con los juicios de relevancia (*qrels*) del corpus.

## 2. Corpus asignado

Cada estudiante recibirá un conjunto de datos. Todos contienen documentos, consultas y qrels. Se utilizarán los identificadores originales de los documentos para que los resultados puedan evaluarse.

| Corpus | Documentos y consultas | Qrels |
| --- | --- | --- |
| NFCorpus | [BeIR/nfcorpus](https://huggingface.co/datasets/BeIR/nfcorpus) | [BeIR/nfcorpus-qrels](https://huggingface.co/datasets/BeIR/nfcorpus-qrels) |
| SciFact | [BeIR/scifact](https://huggingface.co/datasets/BeIR/scifact) | [BeIR/scifact-qrels](https://huggingface.co/datasets/BeIR/scifact-qrels) |

## 3. Requisitos del sistema

### 3.1. Preparación e indexación

El sistema deberá:

1. Cargar e indexar los documentos del corpus asignado.
2. Aplicar y documentar las decisiones de preprocesamiento: tokenización, normalización, eliminación de stopwords y stemming o lematización.
3. Construir un vocabulario y un índice invertido que permita recuperar documentos.
4. Conservar la relación entre cada documento indexado y su identificador original.

La unidad de evaluación será el documento original, porque los qrels relacionan consultas con identificadores de documentos. No se debe dividir el texto en fragmentos.

### 3.2. Modelos de recuperación

Para cualquier consulta de texto, el sistema deberá producir dos rankings independientes:

- TF-IDF con similitud coseno.
- BM25.

Cada ranking deberá ordenar los documentos por puntuación de relevancia y permitir solicitar los primeros `k` resultados.

### 3.3. Servicio web

El sistema deberá estar publicado en internet y ofrecer, como mínimo, los siguientes endpoints:

**Comprobación de disponibilidad**

```http
GET /health
```

**Búsqueda**

```http
POST /search
Content-Type: application/json
```

Ejemplo de solicitud:

```json
{
  "query": "texto de la consulta",
  "model": "bm25",
  "top_k": 10
}
```

El campo `model` deberá aceptar `bm25` y `tfidf`. Una respuesta tendrá esta estructura:

```json
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "identificador_original",
      "score": 8.42,
      "title": "Título, si está disponible",
      "snippet": "Fragmento del documento recuperado"
    }
  ]
}
```

Los resultados deberán estar ordenados de mayor a menor puntuación, no contener documentos duplicados y respetar el valor solicitado de `top_k`. El servicio debe aceptar consultas que el estudiante no conozca de antemano.

## 4. Evaluación de la recuperación

Cada estudiante debe evaluar ambos modelos con consultas de la partición `test` y sus qrels. Presentará, como mínimo:

- Precision@10.
- Recall@10.
- NDCG@10.
- Una comparación de los resultados de TF-IDF y BM25.

En el Jupyter Notebook se deberá indicar qué consultas se utilizaron y permitir reproducir los cálculos. La evaluación del docente podrá utilizar otras consultas del mismo conjunto.

## 5. Prueba del docente

El docente accederá a la URL entregada y realizará solicitudes desde un cliente de web service para comprobar:

1. La disponibilidad del servicio.
2. El funcionamiento de TF-IDF y BM25.
3. El formato, orden y validez de los resultados.
4. La correspondencia de los `document_id` con el corpus y los qrels.
5. La respuesta ante consultas no incluidas en los ejemplos de entrega.

## 6. Entregables

Cada estudiante entregará:

1. La URL pública del servicio web, accesible durante el período de revisión.
2. El código en Jupyter Notebook con los resultados de la ejecución de cada bloque de código, incluyendo los resultados de las métricas de evaluación de los modelos de recuperación.

Se evaluará que ambos modelos funcionen correctamente y que cada estudiante interprete sus resultados.

Pablo Pacheco — SciFact

Uso el título y resumen completos de cada documento y conservo su ID.

Instalación: `python -m pip install -r requirements.txt`. La implementación se guarda en app.py para publicarla.

In [1]:
import sys, json, csv, zipfile, hashlib
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from IPython.display import display
print(sys.version)
ROOT = Path.cwd()
assert (ROOT / "scifact/scifact/corpus.jsonl").exists(), "Ejecuta desde la carpeta E1B"

3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]


Verifico los archivos y los qrels. La evaluación usa las 300 consultas test.

In [2]:
DATA = ROOT / "scifact/scifact"
if (ROOT / "scifact.zip").is_file():
    with zipfile.ZipFile(ROOT / "scifact.zip") as archive:
        assert archive.testzip() is None
        for name in archive.namelist():
            if not name.endswith("/"):
                assert (ROOT / "scifact" / name).read_bytes() == archive.read(name)
    print("ZIP íntegro y extracción idéntica.")
documents = [json.loads(line) for line in (DATA / "corpus.jsonl").read_text(encoding="utf-8").splitlines()]
queries = [json.loads(line) for line in (DATA / "queries.jsonl").read_text(encoding="utf-8").splitlines()]
doc_ids = {d["_id"] for d in documents}
query_ids = {q["_id"] for q in queries}
assert len(doc_ids) == len(documents) == 5183
assert len(query_ids) == len(queries) == 1109
assert all(isinstance(i, str) for i in doc_ids | query_ids)
stats = []
for split in ("train", "test"):
    with (DATA / "qrels" / (split + ".tsv")).open(encoding="utf-8") as handle:
        judgments = list(csv.DictReader(handle, delimiter="	"))
    assert len({(r["query-id"], r["corpus-id"]) for r in judgments}) == len(judgments)
    assert all(r["query-id"] in query_ids and r["corpus-id"] in doc_ids for r in judgments)
    stats.append({"partición": split, "consultas": len({r["query-id"] for r in judgments}),
                  "juicios": len(judgments), "grados": sorted({int(r["score"]) for r in judgments})})
print("Documentos", len(documents), "Consultas", len(queries))
display(pd.DataFrame(stats))
display(pd.DataFrame(documents[:2])[["_id", "title", "text"]])
display(pd.DataFrame(queries[:3]))
hashes = {p.relative_to(DATA).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
          for p in sorted(DATA.rglob("*")) if p.is_file()}
display(pd.DataFrame(hashes.items(), columns=["archivo", "sha256"]))

ZIP íntegro y extracción idéntica.
Documentos 5183 Consultas 1109


,partición,consultas,juicios,grados
0,train,809,919,[1]
1,test,300,339,[1]


,_id,title,text
0,4983,Microstructural development of human newborn c...,Alterations of the architecture of cerebral wh...
1,5836,Induction of myelodysplasia by myeloid-derived...,Myelodysplastic syndromes (MDS) are age-depend...


,_id,text,metadata
0,0,0-dimensional biomaterials lack inductive prop...,{}
1,2,1 in 5 million in UK have abnormal PrP positiv...,"{'13734012': [{'sentences': [4], 'label': 'CON..."
2,4,1-1% of colorectal cancer patients are diagnos...,{}


,archivo,sha256
0,corpus.jsonl,dec31c8182f3d744c7d2c09423756fd1d17cbef75808db...
1,qrels/test.tsv,0864bb985e0ca2367ba217977e72004d549054b2b06666...
2,qrels/train.tsv,a53f2114831916c096b6c37d9e54da68cef4efdcdbd5ed...
3,queries.jsonl,8ff84a7c903f722981cd8d595c022660140c51867b2760...


Aplico NFKC, minúsculas, tokenización y stopwords inglesas, conservando negaciones, números y términos compuestos. Uso Porter para reducir variantes de palabras. El índice guarda documentos y frecuencias por término.

TF-IDF usa TF logarítmico, IDF suavizado y coseno. BM25 usa k1=1,2 y b=0,75, fijados antes de evaluar. Devuelvo hasta k coincidencias positivas, con empates por orden del corpus.

In [3]:
%%writefile app.py
"""SciFact: índice invertido y dos rankings léxicos independientes."""
import json
import re
import unicodedata
from collections import Counter
from functools import lru_cache
from pathlib import Path

import numpy as np
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

ROOT = Path(__file__).resolve().parent
DATA = ROOT / "scifact" / "scifact"
STOPWORDS = ENGLISH_STOP_WORDS - {"no", "not", "nor", "without", "against"}
TOKEN = re.compile(r"[^\W_]+(?:[.\-'][^\W_]+)*|[%+−±<>≤≥=]", re.UNICODE)
STEMMER = PorterStemmer()


@lru_cache(maxsize=100000)
def stem(token):
    return STEMMER.stem(token) if token.isascii() and token.isalpha() else token


def preprocess(text):
    normalized = unicodedata.normalize("NFKC", text).lower().replace("’", "'")
    return [stem(token) for token in TOKEN.findall(normalized) if token not in STOPWORDS]


def load_records(path):
    with Path(path).open(encoding="utf-8") as handle:
        records = [json.loads(line) for line in handle if line.strip()]
    ids = [row["_id"] for row in records]
    if len(ids) != len(set(ids)) or not all(isinstance(i, str) for i in ids):
        raise ValueError("Los identificadores deben ser cadenas únicas")
    return records


class SearchEngine:
    def __init__(self, documents):
        self.documents = documents
        self.ids = [d["_id"] for d in documents]
        if len(self.ids) != len(set(self.ids)):
            raise ValueError("Identificadores duplicados")
        self.n = len(documents)
        if not self.n:
            raise ValueError("Corpus vacío")
        self.postings = {}
        self.lengths = np.zeros(self.n)
        for row, document in enumerate(documents):
            counts = Counter(preprocess(document.get("title", "") + " " + document["text"]))
            self.lengths[row] = sum(counts.values())
            for term, frequency in counts.items():
                self.postings.setdefault(term, []).append((row, frequency))
        self.vocabulary = {term: i for i, term in enumerate(sorted(self.postings))}
        self.df = {term: len(posting) for term, posting in self.postings.items()}
        self.avgdl = float(self.lengths.mean())
        self.idf = {term: 1 + np.log((self.n + 1) / (df + 1)) for term, df in self.df.items()}
        self.bm25_idf = {term: np.log1p((self.n - df + .5) / (df + .5))
                         for term, df in self.df.items()}
        self.norms = np.zeros(self.n)
        for term, posting in self.postings.items():
            for row, frequency in posting:
                self.norms[row] += ((1 + np.log(frequency)) * self.idf[term]) ** 2
        self.norms = np.sqrt(self.norms)
        self.k1, self.b = 1.2, .75
        self.pivot = self.k1 * (1 - self.b + self.b * self.lengths / max(self.avgdl, 1e-12))

    def scores(self, query, model):
        if model not in ("tfidf", "bm25"):
            raise ValueError("Modelo inválido")
        counts = Counter(preprocess(query))
        scores = np.zeros(self.n)
        query_norm_squared = 0.
        for term, frequency in counts.items():
            posting = self.postings.get(term)
            if posting is None:
                continue
            rows, tf = np.array(posting).T
            if model == "tfidf":
                query_weight = (1 + np.log(frequency)) * self.idf[term]
                query_norm_squared += query_weight ** 2
                scores[rows] += query_weight * (1 + np.log(tf)) * self.idf[term]
            else:
                scores[rows] += self.bm25_idf[term] * tf * (self.k1 + 1) / (tf + self.pivot[rows])
        if model == "tfidf":
            denominator = np.sqrt(query_norm_squared) * self.norms
            np.divide(scores, denominator, out=scores, where=denominator > 0)
        return scores

    def search(self, query, model, top_k=10):
        scores = self.scores(query, model)
        # Solo documentos que comparten términos. Los empates siguen el orden del corpus.
        candidates = np.flatnonzero(scores > 0)
        order = candidates[np.lexsort((candidates, -scores[candidates]))][:top_k]
        return [dict(rank=rank, document_id=self.ids[row], score=float(scores[row]),
                     title=self.documents[row].get("title", ""),
                     snippet=self.documents[row]["text"][:350])
                for rank, row in enumerate(order, 1)]

"""API REST. El corpus se indexa una vez al iniciar cada proceso."""
import os
from flask import Flask, jsonify, redirect, request



def create_app(engine=None):
    app = Flask(__name__)
    app.config["MAX_CONTENT_LENGTH"] = 1024 * 1024
    engine = engine or SearchEngine(load_records(DATA / "corpus.jsonl"))

    @app.get("/")
    def index():
        return redirect("/docs")

    @app.get("/docs")
    def docs():
        return """<!doctype html>
<html lang="es">
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>SciFact · API</title>
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/swagger-ui-dist@5.31.0/swagger-ui.css">
<div id="swagger-ui"></div>
<script src="https://cdn.jsdelivr.net/npm/swagger-ui-dist@5.31.0/swagger-ui-bundle.js"></script>
<script>SwaggerUIBundle({url: "/openapi.json", dom_id: "#swagger-ui", validatorUrl: null});</script>
</html>"""

    @app.get("/openapi.json")
    def openapi():
        return jsonify({
            "openapi": "3.0.3",
            "info": {"title": "SciFact · Examen RI", "version": "1.0.0"},
            "paths": {
                "/health": {"get": {"summary": "Comprobar disponibilidad",
                                    "responses": {"200": {"description": "Servicio disponible"}}}},
                "/search": {"post": {
                    "summary": "Buscar documentos",
                    "requestBody": {"required": True, "content": {"application/json": {
                        "schema": {"type": "object", "required": ["query", "model", "top_k"],
                                   "properties": {
                                       "query": {"type": "string", "minLength": 1},
                                       "model": {"type": "string", "enum": ["bm25", "tfidf"]},
                                       "top_k": {"type": "integer", "minimum": 1}}},
                        "example": {"query": "DNA damage and cancer", "model": "bm25", "top_k": 10}
                    }}},
                    "responses": {"200": {"description": "Documentos ordenados por puntuación"},
                                  "400": {"description": "Solicitud inválida"}}
                }}
            }
        })

    @app.get("/health")
    def health():
        return jsonify(status="ok", corpus="SciFact", documents=engine.n)

    @app.post("/search")
    def search():
        payload = request.get_json(silent=True)
        if not isinstance(payload, dict):
            return jsonify(error="Se requiere un objeto JSON"), 400
        query, model, top_k = (payload.get(key) for key in ("query", "model", "top_k"))
        if not isinstance(query, str) or not query.strip():
            return jsonify(error="query debe ser texto no vacío"), 400
        if model not in ("tfidf", "bm25"):
            return jsonify(error="model debe ser tfidf o bm25"), 400
        if type(top_k) is not int or top_k < 1:
            return jsonify(error="top_k debe ser un entero positivo"), 400
        return jsonify(model=model, results=engine.search(query, model, min(top_k, engine.n)))

    return app


engine = SearchEngine(load_records(DATA / "corpus.jsonl"))
app = create_app(engine)


if __name__ == "__main__":
    from waitress import serve
    port = int(os.environ.get("PORT", "8000"))
    print(f"SciFact disponible en http://127.0.0.1:{port}", flush=True)
    serve(app, host="0.0.0.0", port=port)


Overwriting app.py


In [4]:
from app import app, engine, SearchEngine, preprocess, load_records
example = "The β-cells are not inactive: DNA damage decreased by 25% in IL-6-treated cells."
print("Original", example)
print("Procesado", preprocess(example))
print("Título y resumen", documents[0]["title"], documents[0]["text"][:180])
print("Primeros tokens", preprocess(documents[0]["title"] + " " + documents[0]["text"])[:45])
print("N", engine.n, "Vocabulario", len(engine.vocabulary), "avgdl", engine.avgdl)
print("Pares término-documento", sum(engine.df.values()))
term = preprocess("inflammation")[0]
display(pd.DataFrame([{"document_id":engine.ids[row], "tf":tf, "longitud":engine.lengths[row]}
                     for row, tf in engine.postings[term][:10]]))
print("Término", term, "DF", engine.df[term], "IDF", engine.idf[term])
display(pd.DataFrame(list(engine.vocabulary.items())[:12], columns=["término", "columna"]))

Original The β-cells are not inactive: DNA damage decreased by 25% in IL-6-treated cells.
Procesado ['β-cells', 'not', 'inact', 'dna', 'damag', 'decreas', '25', '%', 'il-6-treated', 'cell']
Título y resumen Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging. Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-
Primeros tokens ['microstructur', 'develop', 'human', 'newborn', 'cerebr', 'white', 'matter', 'assess', 'vivo', 'diffus', 'tensor', 'magnet', 'reson', 'imag', 'alter', 'architectur', 'cerebr', 'white', 'matter', 'develop', 'human', 'brain', 'affect', 'cortic', 'develop', 'result', 'function', 'disabl', 'line', 'scan', 'diffusion-weighted', 'magnet', 'reson', 'imag', 'mri', 'sequenc', 'diffus', 'tensor', 'analysi', 'appli', 'measur', 'appar', 'diffus', 'coeffici', 'calcul']
N 

,document_id,tf,longitud
0,5836,1,150.0
1,97884,1,130.0
2,496873,1,145.0
3,712078,1,146.0
4,854417,1,128.0
5,883747,2,102.0
6,888896,4,143.0
7,935034,2,110.0
8,1022115,3,152.0
9,1049501,1,119.0


Término inflamm DF 217 IDF 4.168837175243022


,término,columna
0,%,0
1,+,1
2,0,2
3,0-1,3
4,0-10,4
5,0-100,5
6,0-12,6
7,0-15,7
8,0-2,8
9,0-20,9


Ejemplo de consulta libre.

In [5]:
query = "DNA damage and inflammatory immune response"
for model in ("tfidf", "bm25"):
    print(model)
    display(pd.DataFrame(engine.search(query, model, 5)))

tfidf


,rank,document_id,score,title,snippet
0,1,21439293,0.248248,Beyond pattern recognition: five immune checkp...,Pattern recognition by the innate immune syste...
1,2,40323148,0.245088,Infection and apoptosis as a combined inflamma...,While inflammatory phagocytosis of microbial p...
2,3,5572127,0.242297,Atm-deficient mice exhibit increased sensitivi...,The role of ataxia telangiectasia mutated (ATM...
3,4,36831892,0.232265,Histone tails: Directing the chromatin respons...,Considerable energetic investment is devoted t...
4,5,38131471,0.231366,Molecular mechanisms of mammalian DNA repair a...,DNA damage is a relatively common event in the...


bm25


,rank,document_id,score,title,snippet
0,1,5572127,18.130667,Atm-deficient mice exhibit increased sensitivi...,The role of ataxia telangiectasia mutated (ATM...
1,2,40323148,15.509467,Infection and apoptosis as a combined inflamma...,While inflammatory phagocytosis of microbial p...
2,3,21439293,15.409365,Beyond pattern recognition: five immune checkp...,Pattern recognition by the innate immune syste...
3,4,44724517,14.248753,Kruppel-like factor 2 is a transcriptional reg...,Although myeloid cell activation is requisite ...
4,5,4380287,13.881687,Response to self antigen imprints regulatory m...,Immune homeostasis in tissues is achieved thro...


P@10 = aciertos/10 y R@10 = aciertos/relevantes. NDCG@10 compara el DCG obtenido con el ideal, usando ganancia 2^grado−1 y descuento log2(posición+1).

Los documentos sin juicio tienen grado cero. Sin relevantes, recall y NDCG valen cero. Promedio las métricas de las 300 consultas test y muestro sus IDs.

In [6]:
"""Evaluación exclusivamente sobre los qrels y consultas de test."""
import csv
from collections import defaultdict
from math import log2
from statistics import mean



def load_qrels(path):
    result = defaultdict(dict)
    with path.open(encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle, delimiter="\t"):
            qid, did, relevance = row["query-id"], row["corpus-id"], int(row["score"])
            if did in result[qid] or relevance < 0:
                raise ValueError("Qrel duplicado o negativo")
            result[qid][did] = relevance
    return dict(result)


def metrics(ranking, judged, k=10):
    retrieved = ranking[:k]
    if len(retrieved) != len(set(retrieved)):
        raise ValueError("Ranking duplicado")
    relevant = {doc for doc, grade in judged.items() if grade > 0}
    hits = len(relevant.intersection(retrieved))
    dcg = sum((2 ** judged.get(doc, 0) - 1) / log2(rank + 1)
              for rank, doc in enumerate(retrieved, 1))
    ideal = sorted(judged.values(), reverse=True)[:k]
    idcg = sum((2 ** grade - 1) / log2(rank + 1) for rank, grade in enumerate(ideal, 1))
    return {"precision@10": hits / k, "recall@10": hits / len(relevant) if relevant else 0.,
            "ndcg@10": dcg / idcg if idcg else 0.}


def evaluate(engine, queries, qrels):
    query_map = {q["_id"]: q["text"] for q in queries}
    assert set(qrels) <= query_map.keys()
    assert all(set(judged) <= set(engine.ids) for judged in qrels.values())
    records = []
    for qid in sorted(qrels, key=int):
        for model in ("tfidf", "bm25"):
            ids = [r["document_id"] for r in engine.search(query_map[qid], model, 10)]
            records.append(dict(query_id=qid, query=query_map[qid], model=model,
                                **metrics(ids, qrels[qid]), top10=" ".join(ids)))
    summary = [{"model": model, **{metric: mean(r[metric] for r in records if r["model"] == model)
                for metric in ("precision@10", "recall@10", "ndcg@10")}}
                for model in ("tfidf", "bm25")]
    return records, summary

In [7]:
qrels = load_qrels(DATA / "qrels/test.tsv")
print("IDs de las consultas evaluadas", ", ".join(sorted(qrels, key=int)))
records, summary = evaluate(engine, queries, qrels)
per_query = pd.DataFrame(records)
display(pd.DataFrame(summary))
display(per_query.head(8))
assert len(records) == 600 and len(qrels) == 300

IDs de las consultas evaluadas 1, 3, 5, 13, 36, 42, 48, 49, 50, 51, 53, 54, 56, 57, 70, 72, 75, 94, 99, 100, 113, 115, 118, 124, 127, 128, 129, 130, 132, 133, 137, 141, 142, 143, 146, 148, 163, 171, 179, 180, 183, 185, 198, 208, 212, 213, 216, 217, 218, 219, 230, 232, 233, 236, 237, 238, 239, 248, 249, 261, 268, 269, 274, 275, 279, 294, 295, 298, 300, 303, 312, 314, 324, 327, 338, 343, 350, 354, 362, 380, 384, 385, 386, 388, 399, 410, 411, 415, 421, 431, 436, 437, 439, 440, 443, 452, 475, 478, 491, 501, 502, 507, 508, 513, 514, 516, 517, 521, 525, 527, 528, 532, 533, 535, 536, 539, 540, 544, 549, 551, 552, 554, 560, 569, 575, 577, 578, 587, 589, 593, 597, 598, 613, 619, 623, 628, 636, 637, 641, 644, 649, 659, 660, 674, 684, 690, 691, 692, 693, 700, 702, 715, 716, 718, 721, 723, 727, 728, 729, 742, 743, 744, 756, 759, 768, 770, 775, 781, 783, 784, 785, 793, 800, 805, 808, 811, 814, 820, 821, 823, 830, 831, 832, 834, 837, 839, 845, 847, 852, 859, 870, 873, 879, 880, 882, 887, 903, 904, 9

,model,precision@10,recall@10,ndcg@10
0,tfidf,0.089000,0.8145,0.643527
1,bm25,0.088667,0.8055,0.668180


,query_id,query,model,precision@10,recall@10,ndcg@10,top10
0,1,0-dimensional biomaterials show inductive prop...,tfidf,0.0,0.0,0.000000,13231899 21257564 18953920 7581911 42421723 36...
1,1,0-dimensional biomaterials show inductive prop...,bm25,0.0,0.0,0.000000,21257564 18953920 7581911 20155713 36480032 13...
2,3,"1,000 genomes project enables mapping of genet...",tfidf,0.1,1.0,0.430677,2739854 23389795 19058822 14717500 4632921 437...
3,3,"1,000 genomes project enables mapping of genet...",bm25,0.1,1.0,0.386853,4414547 19058822 4378885 13519661 14717500 210...
4,5,1/2000 in UK have abnormal PrP positivity.,tfidf,0.1,1.0,0.630930,17333231 13734012 42240424 18617259 695938 129...
5,5,1/2000 in UK have abnormal PrP positivity.,bm25,0.1,1.0,1.000000,13734012 18617259 17333231 42240424 695938 129...
6,13,5% of perinatal mortality is due to low birth ...,tfidf,0.0,0.0,0.000000,7662395 17450673 5487448 12779444 1263446 2518...
7,13,5% of perinatal mortality is due to low birth ...,bm25,0.0,0.0,0.000000,7662395 17450673 1263446 25182647 5487448 1979...


En la consulta 72, BM25 coloca el relevante primero y TF-IDF cuarto. En la 232, TF-IDF lo coloca primero y BM25 tercero. En la 1 ambos fallan en top-10 por la poca coincidencia de vocabulario. Las tablas muestran frecuencias y longitudes para comparar los casos.

In [8]:
query_map = {q["_id"]:q["text"] for q in queries}
for qid in ("72", "232", "1"):
    print("Consulta", qid, query_map[qid], "Relevantes", qrels[qid])
    display(per_query[per_query.query_id == qid])
    chosen = set(qrels[qid])
    for model in ("tfidf", "bm25"):
        chosen.add(engine.search(query_map[qid], model, 1)[0]["document_id"])
    evidence = []
    for did in sorted(chosen):
        row = engine.ids.index(did)
        counts = Counter(preprocess(documents[row]["title"] + " " + documents[row]["text"]))
        matches = {t:counts[t] for t in set(preprocess(query_map[qid])) if counts[t]}
        evidence.append({"ID":did, "título":documents[row]["title"], "longitud":engine.lengths[row],
                         "norma TF-IDF":engine.norms[row], "TF coincidentes":matches})
    display(pd.DataFrame(evidence))

Consulta 72 Activator-inhibitor pairs are provided dorsally by Admpchordin. Relevantes {'6076903': 1}


,query_id,query,model,precision@10,recall@10,ndcg@10,top10
30,72,Activator-inhibitor pairs are provided dorsall...,tfidf,0.1,1.0,0.430677,2147704 15953181 10937190 6076903 14492964 139...
31,72,Activator-inhibitor pairs are provided dorsall...,bm25,0.1,1.0,1.000000,6076903 13963620 15953181 41774099 14333540 21...


,ID,título,longitud,norma TF-IDF,TF coincidentes
0,2147704,The homeobox gene goosecoid controls cell migr...,85.0,55.037826,{'dorsal': 3}
1,6076903,Regulation of ADMP and BMP2/4/7 at Opposite Em...,112.0,66.557307,"{'provid': 1, 'dorsal': 3}"


Consulta 232 Cataract and trachoma are the primary cause of blindness in Southern Sudan. Relevantes {'10536636': 1}


,query_id,query,model,precision@10,recall@10,ndcg@10,top10
102,232,Cataract and trachoma are the primary cause of...,tfidf,0.1,1.0,1.0,10536636 7655029 29090823 457630 34760396 2593...
103,232,Cataract and trachoma are the primary cause of...,bm25,0.1,1.0,0.5,7655029 29090823 10536636 34760396 21914176 45...


,ID,título,longitud,norma TF-IDF,TF coincidentes
0,10536636,Prevalence and Causes of Blindness and Low Vis...,42.0,39.500038,"{'southern': 3, 'sudan': 3, 'caus': 2, 'blind'..."
1,7655029,The Burden of Trachoma in Ayod County of South...,218.0,90.068553,"{'southern': 3, 'sudan': 3, 'trachoma': 14, 'c..."


Consulta 1 0-dimensional biomaterials show inductive properties. Relevantes {'31715818': 1}


,query_id,query,model,precision@10,recall@10,ndcg@10,top10
0,1,0-dimensional biomaterials show inductive prop...,tfidf,0.0,0.0,0.0,13231899 21257564 18953920 7581911 42421723 36...
1,1,0-dimensional biomaterials show inductive prop...,bm25,0.0,0.0,0.0,21257564 18953920 7581911 20155713 36480032 13...


,ID,título,longitud,norma TF-IDF,TF coincidentes
0,13231899,In situ regulation of DC subsets and T cells m...,170.0,64.928665,{'biomateri': 1}
1,21257564,How long will long-term potentiation last?,99.0,52.589104,"{'properti': 2, 'induct': 2}"
2,31715818,New opportunities: the use of nanotechnologies...,60.0,41.555902,{}


Compruebo los endpoints, ambos modelos, top-k, consultas nuevas y entradas inválidas.

In [9]:
from threading import Thread
from werkzeug.serving import make_server
import requests
import logging

logging.getLogger("werkzeug").setLevel(logging.ERROR)

def check_service(url):
    response = requests.get(url + "/health", timeout=150)
    assert response.status_code == 200 and response.json()["documents"] == 5183
    for model in ("tfidf", "bm25"):
        for query in ("DNA damage and cancer", "How does inflammation affect immune cells?", "zzunknownterm"):
            for k in (1, 10):
                body = {"query": query, "model": model, "top_k": k}
                response = requests.post(url + "/search", json=body, timeout=30)
                assert response.status_code == 200
                payload = response.json()
                expected = engine.search(query, model, k)
                assert payload["model"] == model and len(payload["results"]) == len(expected)
                for actual, local in zip(payload["results"], expected):
                    assert np.isclose(actual["score"], local["score"])
                    assert {key: value for key, value in actual.items() if key != "score"} == {
                        key: value for key, value in local.items() if key != "score"
                    }
    for body in ({}, {"query":"", "model":"bm25", "top_k":10},
                 {"query":"cells", "model":"invalid", "top_k":10},
                 {"query":"cells", "model":"tfidf", "top_k":0}):
        assert requests.post(url + "/search", json=body, timeout=30).status_code == 400
    print("API verificada: ambos modelos, top-k, consulta nueva y errores.")

assert metrics(["a", "x"], {"a":1}) == {"precision@10":0.1, "recall@10":1.0, "ndcg@10":1.0}
server = make_server("127.0.0.1", 0, app)
worker = Thread(target=server.serve_forever, daemon=True)
worker.start()
try:
    check_service(f"http://127.0.0.1:{server.server_port}")
finally:
    server.shutdown()
    server.server_close()
    worker.join(timeout=5)


API verificada: ambos modelos, top-k, consulta nueva y errores.


Servicio publicado en [Render](https://scifact-api.onrender.com/docs).

In [10]:
PUBLIC_URL = "https://scifact-api.onrender.com"
check_service(PUBLIC_URL)

API verificada: ambos modelos, top-k, consulta nueva y errores.


TF-IDF obtiene un recall algo mayor (0,8145 frente a 0,8055). BM25 obtiene mayor NDCG (0,6682 frente a 0,6435), por las posiciones de los relevantes. La precisión es cercana a 0,089 en ambos, con pocos documentos relevantes por consulta.

Datos de [BEIR SciFact](https://huggingface.co/datasets/BeIR/scifact), CC BY-SA 4.0.